![BasQ banner](logo_cropped.png)

# <b>Combinatorial Optimization with QAOA: Max-Cut </b>
**Author:** Benjamin Tirado  
**Created for:** BasQ Qiskit Fall Fest 2026 — Basque Quantum (BasQ)

<a id="goal"></a>
<div class="alert alert-block alert-success">
    
<b>Goal of this notebook: </b> Solve the **Max-Cut problem** on a small graph using the **Quantum Approximate Optimization Algorithm (QAOA)**. We will translate a combinatorial problem into a **cost Hamiltonian**, build the QAOA circuit from alternating **cost** and **mixer** layers, optimize its parameters with a classical routine, and sample the result to read out a good partition — checking it against a brute-force solution. By the end you will have a reusable QAOA template that you can point at harder, more realistic optimization problems.
</div>

# <b>Table of Contents</b>

* [Background](#background)
* [Pre-requisites](#prereq)
* [Defining the Problem: Max-Cut on a Graph](#problem)
* [From Cost Function to Hamiltonian](#hamiltonian)
* [Building the QAOA Circuit](#circuit)
* [Optimizing the Parameters](#optimize)
* [Sampling and Reading the Solution](#sample)
* [Results and Evaluation](#res)
* [Moving forward](#move)
* [Useful resources](#use)

## <b>Background </b> <a id="background"></a>

Many important problems in logistics, finance, scheduling, and network design are **combinatorial optimization** problems: choose the best option out of an enormous, discrete set of possibilities. Because the number of candidate solutions grows exponentially with problem size, exact classical methods struggle as problems get large, and much of computer science is devoted to good approximate methods.

Quantum computers offer a different angle of attack. The **Quantum Approximate Optimization Algorithm (QAOA)**, introduced by Farhi, Goldstone and Gutmann in 2014, is a hybrid quantum-classical algorithm designed to find good approximate solutions to such problems. It has since become one of the most studied quantum optimization methods, and a decade later it sits at the heart of utility-scale demonstrations on IBM hardware (for example the error-suppressed pipeline of Sachdeva et al., 2024, which tackled nontrivial binary optimization at the 100+ qubit scale).

The idea behind QAOA is elegant. Any problem where we want to *minimize a cost* can be encoded into a **cost Hamiltonian** $\hat{H}_C$, built so that its lowest-energy state corresponds to the best solution. QAOA prepares a trial state by alternating two ingredients:

$$
|\psi(\vec{\gamma}, \vec{\beta})\rangle = \prod_{k=1}^{p} e^{-i\beta_k \hat{H}_M}\, e^{-i\gamma_k \hat{H}_C}\; |+\rangle^{\otimes n},
$$

where $\hat{H}_C$ is the **cost** Hamiltonian (it rewards good solutions), $\hat{H}_M = \sum_i X_i$ is the **mixer** (it explores different candidate solutions), and the integer $p$ is the number of **layers** (also called the QAOA depth). The angles $\vec{\gamma}$ and $\vec{\beta}$ are tuned by a classical optimizer to minimize the expected cost. As $p$ grows, QAOA can represent better and better solutions, at the price of deeper circuits.

As our teaching problem we use **Max-Cut**, the standard entry point to QAOA: it maps onto a cost Hamiltonian in the simplest possible way, and on small graphs we can check the quantum answer against brute force.

# <b>Pre-requisites </b> <a id="prereq"></a>
For starters, make sure you have installed the Qiskit SDK and its supporting modules: `qiskit_aer` (for noiseless simulations) and `qiskit_ibm_runtime` (for real-hardware experiments), as well as the visualization package `qiskit[visualization]`. We use `qiskit.circuit.library.QAOAAnsatz` to build the QAOA circuit, the built-in `StatevectorEstimator` and `StatevectorSampler` primitives to evaluate and sample it, and `scipy.optimize` for the classical parameter optimization. Graphs are handled with `rustworkx`, Qiskit's companion graph library.

In [ ]:
# %pip install qiskit qiskit_aer qiskit_ibm_runtime qiskit[visualization] rustworkx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
from itertools import product

from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from scipy.optimize import minimize

# <b>Defining the Problem: Max-Cut on a Graph </b> <a id="problem"></a>
The **Max-Cut** problem is easy to state. Given a graph of nodes connected by edges, split the nodes into **two groups** so that the number of edges running *between* the groups (the edges we "cut") is as large as possible.

We describe a partition by assigning each node a label $x_i \in \{0, 1\}$ — which side of the cut it is on. An edge $(i, j)$ is cut exactly when its two endpoints have different labels. The number of cut edges is therefore

$$
C(\vec{x}) = \sum_{(i,j)\in E} \big[\, x_i \ne x_j \,\big],
$$

and Max-Cut asks us to maximize this over all $2^n$ possible labellings. Let us build a small graph to work with — five nodes, chosen so we can still check every possibility by hand.

In [ ]:
# Build a small 5-node graph
num_nodes = 5
edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]

graph = rx.PyGraph()
graph.add_nodes_from(range(num_nodes))
graph.add_edges_from([(i, j, 1.0) for i, j in edges])  # weight 1 on every edge

draw_graph(graph, node_size=600, with_labels=True, node_color="#9ecae1")

# <b>From Cost Function to Hamiltonian </b> <a id="hamiltonian"></a>
To run Max-Cut on a quantum computer we translate the cost function into a **cost Hamiltonian** whose ground state encodes the optimal cut. The standard trick is to replace each binary variable $x_i \in \{0,1\}$ by a spin $Z_i \in \{+1, -1\}$ via $x_i = (1 - Z_i)/2$. Substituting this into the Max-Cut objective, each edge $(i,j)$ contributes

$$
\big[\, x_i \ne x_j \,\big] \;\longrightarrow\; \tfrac{1}{2}\big(1 - Z_i Z_j\big).
$$

The term $Z_i Z_j$ equals $+1$ when both endpoints are on the same side (edge not cut) and $-1$ when they differ (edge cut). Because quantum systems naturally settle into **minimum** energy, and we want to **maximize** the cut, we minimize the following cost Hamiltonian, dropping constant offsets:

$$
\hat{H}_C = \sum_{(i,j)\in E} \tfrac{1}{2}\, Z_i Z_j.
$$

Its ground state is the spin configuration — the partition — that cuts the most edges. We build it below as a `SparsePauliOp`, one Pauli-`ZZ` string per edge.

In [ ]:
def maxcut_cost_hamiltonian(num_nodes, edges):
    """Build the Max-Cut cost Hamiltonian sum_(i,j) 0.5 * Z_i Z_j."""
    terms = []
    for i, j in edges:
        pauli = ["I"] * num_nodes
        pauli[i] = "Z"
        pauli[j] = "Z"
        # Qiskit reads Pauli strings right-to-left (qubit 0 is the rightmost char)
        terms.append(("".join(reversed(pauli)), 0.5))
    return SparsePauliOp.from_list(terms)


cost_hamiltonian = maxcut_cost_hamiltonian(num_nodes, edges)
print("Cost Hamiltonian (one ZZ term per edge):")
print(cost_hamiltonian)

# <b>Building the QAOA Circuit </b> <a id="circuit"></a>
With the cost Hamiltonian in hand, Qiskit can build the QAOA circuit for us. The `QAOAAnsatz` takes the cost operator and a number of repetitions `reps` (this is the depth $p$) and assembles the alternating pattern automatically:

- it starts every qubit in the equal superposition $|+\rangle$ (all candidate solutions at once),
- applies the **cost layer** $e^{-i\gamma_k \hat{H}_C}$ (which imprints the problem structure), and
- applies the **mixer layer** $e^{-i\beta_k \hat{H}_M}$ with $\hat{H}_M = \sum_i X_i$ (which shuffles amplitude between candidates),

repeating for $p$ layers. This leaves $2p$ free parameters (one $\gamma$ and one $\beta$ per layer) for the optimizer to tune.

We start with the shallowest non-trivial case, $p = 1$, which has just two parameters. We add measurements so we can later sample bitstrings from the circuit.

In [ ]:
p = 1  # QAOA depth (number of layers)
ansatz = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=p)
ansatz.measure_all()

print(f"QAOA depth p = {p}")
print("Number of qubits:", ansatz.num_qubits)
print("Number of parameters:", ansatz.num_parameters)
ansatz.decompose(reps=3).draw("mpl", fold=-1)

# <b>Optimizing the Parameters </b> <a id="optimize"></a>
QAOA is a **hybrid** algorithm: the quantum computer prepares the trial state and estimates its energy, while a **classical optimizer** adjusts the angles $(\vec{\gamma}, \vec{\beta})$ to drive that energy down. The quantity we minimize is the expected value of the cost Hamiltonian,

$$
E(\vec{\gamma}, \vec{\beta}) = \langle \psi(\vec{\gamma}, \vec{\beta}) | \hat{H}_C | \psi(\vec{\gamma}, \vec{\beta}) \rangle,
$$

which we evaluate with the `StatevectorEstimator` (exact and noiseless — ideal for learning). We wrap this in a small cost function and hand it to `scipy.optimize.minimize` with the **COBYLA** optimizer, a gradient-free method that copes well with the noisy, non-convex landscapes typical of QAOA.

Note we pass the ansatz *without* its measurements to the Estimator (expectation values are computed from the statevector, not from samples); the measured copy is kept for the sampling step that follows.

In [ ]:
estimator = StatevectorEstimator()

# Estimator needs the circuit without measurements
ansatz_no_meas = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=p)

objective_history = []

def cost_function(params):
    """Expected energy of the cost Hamiltonian for given QAOA angles."""
    pub = (ansatz_no_meas, cost_hamiltonian, params)
    energy = estimator.run([pub]).result()[0].data.evs
    energy = float(energy)
    objective_history.append(energy)
    return energy

# A reasonable starting point; QAOA is sensitive to initialization.
np.random.seed(42)
initial_params = np.random.uniform(0, np.pi, ansatz_no_meas.num_parameters)

result = minimize(cost_function, initial_params, method="COBYLA",
                  options={"maxiter": 200})

print("Optimization finished.")
print("Optimal parameters:", np.round(result.x, 4))
print(f"Minimum expected energy: {result.fun:.4f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(objective_history, color="#1f77b4")
plt.xlabel("Optimizer iteration")
plt.ylabel(r"Expected cost  $\langle H_C \rangle$")
plt.title("QAOA optimization convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# <b>Sampling and Reading the Solution </b> <a id="sample"></a>
Minimizing the energy gives us good angles, but the *solution* we care about is a **bitstring** — an actual assignment of nodes to the two sides of the cut. To get it, we bind the optimal parameters into the measured circuit and **sample** it many times with the `StatevectorSampler`. The most frequently observed bitstrings are QAOA's candidate solutions.

For each sampled bitstring we compute how many edges it cuts, and keep the best. On a small graph the optimal partition should dominate the sampled distribution when QAOA has done its job.

In [ ]:
def cut_value(bitstring, edges):
    """Number of edges cut by a given assignment (bitstring)."""
    # bitstring[i] is the side (0 or 1) assigned to node i
    return sum(1 for i, j in edges if bitstring[i] != bitstring[j])


sampler = StatevectorSampler()
best = ansatz.assign_parameters(result.x)
samples = sampler.run([best], shots=4096).result()[0].data.meas.get_counts()

# Convert each measured key (a bit-string) into a per-node assignment.
# Qiskit bitstrings are little-endian: the rightmost char is qubit 0.
def key_to_assignment(key, num_nodes):
    bits = key.replace(" ", "")[::-1]  # reverse -> index 0 is node 0
    return [int(bits[i]) for i in range(num_nodes)]

scored = []
for key, count in samples.items():
    assignment = key_to_assignment(key, num_nodes)
    scored.append((assignment, count, cut_value(assignment, edges)))

# Sort by how often each bitstring appeared
scored.sort(key=lambda t: t[1], reverse=True)
print("Most sampled bitstrings (assignment, count, cut value):")
for assignment, count, cut in scored[:5]:
    print(f"  {assignment}   count={count:5d}   cut={cut}")

# <b>Results and Evaluation </b> <a id="res"></a>
Because the graph is small, we can **brute-force** the true optimum: enumerate all $2^n$ partitions, compute each cut value, and find the maximum. Comparing QAOA's best sampled cut to this exact optimum tells us how well the algorithm did.

A common summary metric is the **approximation ratio**, the ratio of the cut QAOA found to the true optimal cut. A ratio of $1.0$ means QAOA recovered an optimal partition. At $p = 1$ on a simple graph you should already get close; one of the first things you will explore in the challenge is how this ratio improves as the depth $p$ grows.

In [ ]:
# Brute-force optimum
best_cut, best_assignments = 0, []
for bits in product([0, 1], repeat=num_nodes):
    c = cut_value(bits, edges)
    if c > best_cut:
        best_cut, best_assignments = c, [bits]
    elif c == best_cut:
        best_assignments.append(bits)

qaoa_best = max(scored, key=lambda t: t[2])
qaoa_cut = qaoa_best[2]
approx_ratio = qaoa_cut / best_cut

print(f"Brute-force optimal cut : {best_cut}")
print(f"QAOA best sampled cut   : {qaoa_cut}")
print(f"Approximation ratio     : {approx_ratio:.3f}")

# Visualize the QAOA partition
colors = ["#f4a582" if b == 0 else "#92c5de" for b in qaoa_best[0]]
draw_graph(graph, node_size=600, with_labels=True, node_color=colors)

# <b>Moving forward </b> <a id="move"></a>
This notebook walked through the full QAOA workflow on a deliberately tiny, unweighted Max-Cut instance: encode the cost, build the alternating circuit, optimize two angles, sample, and check against brute force. The hackathon challenge takes QAOA out of the toy graph and points it at a **real-world optimization problem** — and, crucially, at problems with **constraints**, which is where encoding becomes an art.

The three levels below are open-ended: they state a target, not a recipe. Choosing the encoding, the depth, the optimizer, and (from the intermediate level) the overall algorithmic approach is part of the challenge.

### <b>The challenge problem — portfolio selection</b>
The challenge centres on a **portfolio-selection** problem: from a universe of assets, choose a subset that maximizes expected return while penalizing risk, subject to a **budget constraint** (for example, pick exactly $K$ assets). This problem is a natural fit for QAOA — it maps to a cost Hamiltonian just like Max-Cut — but it adds the essential ingredient of a constraint, which you must fold into the encoding (typically as a penalty term). Small instances can still be checked by brute force, so you can measure how well you are doing.

### <b>Getting Started — Beginner</b>
Set up a small portfolio-selection instance, encode it as a cost Hamiltonian (objective **plus** a budget-constraint penalty), and solve it with QAOA following the workflow from this notebook. Then study the two knobs that matter most at this level: how the **approximation ratio improves with QAOA depth $p$**, and how the **penalty strength** on the constraint affects whether the solutions QAOA returns are actually feasible. Verify against a brute-force optimum.

### <b>Intermediate — Choose Your Approach</b>
Scale to a larger or more tightly constrained instance and commit to a strategy. This level **forks**, and part of the task is to justify which path you take and compare it against a sensible classical baseline:

- **Variational / QAOA route.** Push QAOA harder: warm-starting from a classical relaxation, constraint-preserving or custom mixers that keep the search inside the feasible set, better classical optimizers, or smarter parameter initialization.
- **Quantum machine learning route.** Treat the problem through a learning lens: for example, a variational model trained to produce high-quality (feasible) solutions, or a QML approach to the underlying selection/classification structure.

### <b>Advanced — Push Toward Utility Scale</b>
Move toward the regime where naive encodings and shallow circuits stop working. Explore what it takes to get good solutions as the problem grows: confronting **trainability** issues (barren plateaus), **higher-order terms or heavier constraints**, and the **error suppression and hardware-aware compilation** that utility-scale optimization demands. The inspiration here is the integrated, error-suppressed pipelines that have pushed gate-model optimization to the 100+ qubit scale — how close can you get to a robust, scalable solver?

### <b>Running on real hardware</b>
Whichever level you tackle, moving from the `StatevectorEstimator` to a real IBM Quantum device (such as `ibm_basquecountry`) changes the game. The two-qubit cost terms must be mapped onto the heavy-hex connectivity, and terms between non-adjacent qubits force the transpiler to insert SWAPs that deepen the circuit — SWAP networks and commuting-gate routing become valuable. Finite shots and gate noise also blur the sampled distribution, so error suppression and mitigation (dynamical decoupling, twirling, and more) become part of getting a trustworthy answer.

# <b>Useful resources </b> <a id="use"></a>

The following resources may be useful when extending this introductory QAOA example toward the full challenge.

### <b>QAOA and quantum optimization in Qiskit </b>

- [Quantum approximate optimization algorithm (IBM tutorial)](https://quantum.cloud.ibm.com/docs/tutorials/quantum-approximate-optimization-algorithm)  
  End-to-end QAOA for Max-Cut, from a small graph to a 100-node utility-scale problem on real hardware — the closest reference to this notebook.

- [Advanced techniques for QAOA](https://quantum.cloud.ibm.com/docs/tutorials/advanced-techniques-for-qaoa)  
  SWAP networks, commuting-gate routing, and depth reduction for QAOA on hardware — relevant to the advanced level.

- [QAOAAnsatz documentation](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.library.QAOAAnsatz)  
  Reference page for the QAOA circuit class used here, including the mixer and initial-state options.

- [Qiskit Optimization documentation](https://qiskit-community.github.io/qiskit-optimization/)  
  Tools for building `QuadraticProgram`s, encoding constraints, and converting problems to Ising Hamiltonians.

- [Max-Cut and TSP tutorial](https://qiskit-community.github.io/qiskit-optimization/tutorials/06_examples_max_cut_and_tsp.html)  
  Shows the QUBO-to-Ising conversion and application classes for standard optimization problems.

- [Portfolio optimization tutorial](https://qiskit-community.github.io/qiskit-finance/tutorials/01_portfolio_optimization.html)  
  Encoding a budget-constrained portfolio-selection problem for variational solvers — directly relevant to the challenge problem.

### <b>Encoding, constraints, and optimizers </b>

- [Converters for quadratic programs](https://qiskit-community.github.io/qiskit-optimization/tutorials/02_converters_for_quadratic_programs.html)  
  How constraints become penalty terms in the cost Hamiltonian — the core encoding skill for this track.

- [Warm-starting QAOA (how-to)](https://qiskit-community.github.io/qiskit-optimization/tutorials/10_warm_start_qaoa.html)  
  Initializing QAOA from a classical relaxation — a useful lever for the intermediate level.

- [SamplerV2 and primitives](https://quantum.cloud.ibm.com/docs/guides/primitives)  
  The Sampler and Estimator primitives used to sample solutions and evaluate the cost.

### <b>Hardware execution, transpilation, and error mitigation </b>

- [Transpile with pass managers](https://quantum.cloud.ibm.com/docs/guides/transpile-with-pass-managers)  
  Transpiling circuits for a backend and its heavy-hex connectivity.

- [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/guides/error-mitigation-and-suppression-techniques)  
  Overview of the resilience options available through Qiskit Runtime.

### <b>Background reading </b>

- [Farhi, Goldstone & Gutmann, *A Quantum Approximate Optimization Algorithm* (2014)](https://arxiv.org/abs/1411.4028)  
  The paper that introduced QAOA — the foundation of this track.

- [Sachdeva et al., *Quantum optimization ... on a 127-qubit gate-model IBM quantum computer* (2024)](https://arxiv.org/abs/2406.01743)  
  An error-suppressed QAOA-style pipeline for nontrivial binary optimization at utility scale — the inspiration for the advanced level.

# <b>Credits and license</b>

This notebook was written by **Benjamin Tirado** for the **BasQ Qiskit Fall Fest 2026**, organised by Basque Quantum (BasQ).

© 2026 Benjamin Tirado. The text, figures and explanations are released under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); the code is released under the
[Apache License 2.0](https://www.apache.org/licenses/LICENSE-2.0), the same license as Qiskit.

You are welcome to run, adapt and share this material — including as a starting point for your own
challenge submission — provided the attribution above is kept. If you reuse it publicly, please cite it as:

> B. Tirado, *Combinatorial Optimization with QAOA: Max-Cut*, tutorial notebook, BasQ Qiskit Fall Fest, 2026.

Built with [Qiskit](https://www.ibm.com/quantum/qiskit), [rustworkx](https://www.rustworkx.org/)
and [SciPy](https://scipy.org/), which remain the property of their respective authors and are used
under their own licenses.

*Questions, corrections or suggestions:* benjamin.tirado@ehu.eus